## Script 3: "Feature Engineering"

In this step, we use LLMs to generate focused text extractions for specific article attributes we care about. These focused summaries serve as concise representations of key features but do not include label or cluster derivation.

By concentrating on individual attributes, we reduce token usage for downstream tasks and provide clearer, more discriminative inputs that improve clustering and classification quality later in the pipeline.

In [ ]:
import fenic as fc

from dotenv import load_dotenv

fc.configure_logging()

load_dotenv()

config = fc.SessionConfig(
        app_name="medium_curation",
        semantic=fc.SemanticConfig(
            language_models={
                "flash": fc.GoogleGLAModelConfig(
                    model_name="gemini-2.0-flash",
                    rpm=2000,
                    tpm=4_000_000,
                ),
            },
            embedding_models={
                "large": fc.OpenAIModelConfig(
                    model_name="text-embedding-3-large",
                    rpm=3000,
                    tpm=1_000_000
                )
            }
        ),
    )

session = fc.Session.get_or_create(config)

## Step 1: Define the prompts for the individual features we want to extract:
- content topic
- presence of code
- technical keywords
- form factor
- technical complexity

In [12]:
TOPIC_MODELING_PROMPT= """
You will be analyzing and summarizing a Medium article. Your task is to distill the main topics and key themes of the article into a concise summary.

Here is the Medium article:
title: {title}
text: {text}

Carefully read and analyze the article, focusing on its core content and central ideas. Then, follow these steps to create your summary:

1. Identify the broad subject area(s) covered in the article. These may include topics such as artificial intelligence, ethics, productivity, education, creativity, software development, or other relevant areas.

2. Determine the specific problem, trend, or idea that the article explores in depth.

3. Identify the main concepts or entities discussed in the article. These could include specific technologies (e.g., ChatGPT, large language models), societal issues (e.g., misinformation), or processes (e.g., upskilling).

4. Synthesize this information into a summary of 3-4 concise sentences. Your summary should:
   - Cover the broad subject area(s)
   - Explain the specific focus of the article
   - Mention the main concepts or entities involved
   - Be as information-dense as possible, avoiding filler words
   - Focus solely on the content and themes of the article
   - NOT include any information about the article's style, structure, or author

5. Conclude your analysis by listing 2-3 keywords or short phrases that best capture the central topics of the article.
"""

CODE_PRESENCE_PROMPT = """
Determine if the article contains code snippets or programming examples.
Title: {title}
Text: {text}

Carefully read through the article and look for the presence of code blocks, technical scripts, or programming examples. These might be formatted differently from the regular text.
"""

TECHNICAL_TERMS_PROMPT = """
You will be analyzing a Medium article to extract a list of specific libraries, frameworks, models, or programming languages mentioned in the text. Your task is to identify and list only well-known and nameable technologies.

Instructions:
1. Carefully read through the article.
2. Identify specific libraries, frameworks, models, or programming languages mentioned in the text.
3. Only include well-known and nameable technologies.
4. Do not include general terms or abstract concepts.

Examples of what to include: PyTorch, LangChain, Claude, ChatGPT, LlamaIndex, LangGraph, MCP

Examples of what to exclude: AI, machine learning, deep learning, neural networks, LLMs, or other abstract and general concepts

Remember to focus only on concrete, nameable technologies and avoid listing general concepts or terms.
"""

FORM_FACTOR_PROMPT = """
Your task is to analyze an article's title and full body text, then write a detailed and focused description that clearly and precisely captures the article's overall style, structure, and organization. This description should be 3-5 sentences long.

Here is the article title:
{title}

Here is the full article body:
{text}

Carefully read and analyze both the title and the full body text. Pay close attention to the following aspects:

1. How the article's content is arranged or sequenced (e.g., step-by-step instructions, numbered or bulleted lists, chronological storytelling, logical argument progression, objective reporting, or simplified concept explanation).

2. The way the article engages or guides the reader (e.g., providing direct how-to guidance, listing discrete tips or points, telling a personal or thematic story, making a persuasive case, summarizing facts neutrally, or breaking down complex topics accessibly).

3. Any distinctive features in tone, voice, or presentation (e.g., practical and instructional, concise and enumerative, narrative and emotive, analytical and persuasive, formal and factual, or clear and explanatory).

4. The level of detail and complexity in the style (e.g., very detailed steps, high-level summaries, rich examples, use of personal anecdotes, or emphasis on clarity over depth).

When writing your description:
- Focus on describing the characteristics and flow of the article's presentation in natural, precise language.
- Avoid using explicit style category names or labels.
- Ensure your description is detailed and focused, clearly capturing the article's overall style, structure, and organization.

Here's an example of what your output should look like:

The article guides readers through a detailed, stepwise process, offering clear instructions and practical examples to teach a specific technical skill. The tone is instructional and straightforward, avoiding personal anecdotes or theoretical discussions. Information is presented sequentially, making it easy for readers to follow and apply the steps.

Now, based on your analysis of the provided article title and body, write your 3-5 sentence description.
"""

TECHNICAL_COMPLEXITY_PROMPT = """
You are tasked with assessing the complexity and technical depth of a Medium article. Your goal is to evaluate the intellectual effort and background knowledge required to fully understand the content in 1-2 concise sentences.

Here is the article body:
{text}

Analyze the complexity and technical depth of this article, focusing on the following criteria:

1. The target audience's familiarity with the subject (e.g., general reader, tech-savvy user, domain practitioner, advanced expert)
2. Any prerequisite knowledge assumed (e.g., none, basic AI concepts, prior coding experience, familiarity with recent research)
3. The depth of reasoning or analysis (e.g., introductory overview, moderate conceptual depth, advanced technical or theoretical complexity)

Use your scratchpad to organize your thoughts and analysis before providing your final assessment:

<scratchpad>
[Use this space to note your observations and analysis based on the criteria above]
</scratchpad>

Now, provide your final assessment of the article's complexity and technical depth in 1-2 concise sentences. Focus solely on describing how cognitively demanding or knowledge-intensive the content is, using clear and neutral language. Do not summarize the topic itself.

Write your assessment inside <assessment> tags.
"""

## Step 2: Extract Features

In [13]:
filtered = session.table("with_on_topic_label").filter(fc.col("is_on_topic")).drop("is_on_topic")

with_topic_annotation = (
    filtered
    .with_column("topic_annotation", fc.semantic.map(TOPIC_MODELING_PROMPT))
    .cache()
)

# use select statement to run mapping and embedding concurrently.
with_code_annotation = (
    with_topic_annotation
    .select(
        "*",
        fc.semantic.predicate(CODE_PRESENCE_PROMPT).alias("has_code"),
        fc.semantic.embed(fc.col("topic_annotation")).alias("topic_embedding")
    )
    .cache()
)

list_output_schema = fc.ExtractSchema([
    fc.ExtractSchemaField(
        name="technical_terms",
        data_type=fc.ExtractSchemaList(element_type=fc.StringType),
        description=TECHNICAL_TERMS_PROMPT
    ),
])
with_technical_terms_annotation = (
    with_code_annotation
    .with_column("technical_terms", fc.semantic.extract(fc.col("text"), list_output_schema).technical_terms)
    .cache()
)
with_form_factor_annotation = (
    with_technical_terms_annotation
    .with_column("form_factor", fc.semantic.map(FORM_FACTOR_PROMPT))
    .cache()
)
with_technical_complexity_annotation = (
    with_form_factor_annotation
    .with_column("technical_complexity", fc.semantic.map(TECHNICAL_COMPLEXITY_PROMPT))
    .cache()
)

## Step 3: Save Results

In [ ]:
with_technical_complexity_annotation.write.save_as_table("with_features")

In [15]:
session.stop()